# FinSafe 05 — 100-step finance SFT and matched validation generations
Run after 04. Preliminary one-seed experiment, not a final research conclusion. Starts a fresh training directory, never extends the five-step smoke adapter. Train only on benign curated finance examples. Saves full-state checkpoints every 10 steps and generates the same finance/safety validation sets. Reserved test data is not evaluated. Safety metrics require reviewed response labels. Uses one T4; expect roughly 15–25 minutes, varying by runtime.

In [ ]:
import sys, os, json, subprocess, time, datetime
from pathlib import Path
started=time.monotonic()
print(f"[{datetime.datetime.now(datetime.timezone.utc).isoformat()}] START stage05 bootstrap",flush=True)
try:
    project=Path("/kaggle/working/finsafe")
    if not project.exists():
        raise RuntimeError("Restore the project/data or run notebook 01 and 02 first")
    status=subprocess.check_output(["git","-C",str(project),"status","--porcelain","--untracked-files=no"],text=True)
    if status.strip():
        raise RuntimeError("Save tracked-file edits before updating")
    subprocess.run(["git","-C",str(project),"pull","--ff-only"],check=True)
    sys.path.insert(0,str(project/"src"))
    from finsafe.progress import Progress
    progress=Progress(project/"runs/notebook-05/logs","notebook-05")
    progress.log(f"DONE bootstrap in {time.monotonic()-started:.1f}s")
except BaseException as error:
    print(f"FAILED bootstrap after {time.monotonic()-started:.1f}s ({type(error).__name__})",flush=True)
    raise


In [ ]:
with progress.stage("01-prerequisites"):
    from kaggle_secrets import UserSecretsClient
    from finsafe.data import read_jsonl
    runtime_env=os.environ.copy()
    runtime_env.update(HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN"),
        HF_HOME="/kaggle/working/.cache/huggingface",HF_NAMESPACE="Finsafe-team",
        CUDA_VISIBLE_DEVICES="0",TOKENIZERS_PARALLELISM="false",TRL_ONLY_MODE="1",
        DISABLE_UNSLOTH_FOR_TRL="1",PYTHONUNBUFFERED="1")
    gpu_py=str(project/".venv-gpu/bin/python")
    cli=str(project/".venv-gpu/bin/finsafe")
    revisions=json.loads((project/"runs/environment/model-revisions.json").read_text())
    global_sha=revisions["CohereLabs/tiny-aya-global"]
    filtered=project/"data/processed/tatqa-tokenized"
    curated=filtered/"curated"
    train_rows=read_jsonl(curated/"sft_messages.jsonl")
    if len(train_rows)<40:
        raise RuntimeError("Too few accepted training examples for the 5-step, effective-batch-8 smoke test")
    progress.log(f"Accepted curated training examples: {len(train_rows)}")
    progress.log((filtered/"token_manifest.json").read_text())
    progress.run([cli,"doctor","--gpu"],cwd=project,env=runtime_env)


In [ ]:
with progress.stage("02-verify-baselines"):
    if not (project/"runs/safety-baseline-pilot-v1/status.json").is_file():raise RuntimeError("Run notebook 04 first")
    if len(read_jsonl(project/"runs/safety-baseline-pilot-v1/predictions.jsonl"))!=36:raise RuntimeError("Incomplete safety baseline")
    progress.log("Original finance and safety validation baselines exist; test prompts stay reserved")


In [ ]:
with progress.stage("03-finance-sft-100"):
    sft_dir=project/"checkpoints/finance-sft-100-v1"
    progress.run([gpu_py,"-u","scripts/train_sft.py","--revision",global_sha,"--data",str(curated),"--out",str(sft_dir),"--max-steps","100","--save-steps","10","--load-in-4bit","--resume"],cwd=project,env=runtime_env)
    import math
    state=json.loads((sft_dir/"completed.json").read_text())
    if state['global_step']!=100 or not math.isfinite(state['metrics']['train_loss']):raise RuntimeError("Training verification failed")
    progress.log(json.dumps(state,indent=2))


In [ ]:
with progress.stage("04-finance-validation-after-sft"):
    finance_predictions=project/"runs/finance-sft-100-v1/predictions.jsonl"
    progress.run([gpu_py,"-u","scripts/generate.py","--model","CohereLabs/tiny-aya-global","--revision",global_sha,"--adapter",str(sft_dir),"--data",str(filtered/"validation.jsonl"),"--condition","finance-sft-100-v1","--out",str(finance_predictions),"--max-new-tokens","64","--load-in-4bit","--resume"],cwd=project,env=runtime_env)
    progress.run([cli,"summarize","--data",str(filtered/"validation.jsonl"),"--predictions",str(finance_predictions),"--out","runs/finance-sft-100-v1/report.json"],cwd=project,env=runtime_env)
    progress.run([cli,"audit-replay","--data",str(filtered/"validation.jsonl"),"--predictions",str(finance_predictions),"--out","runs/finance-sft-100-v1/auditkit.json"],cwd=project,env=runtime_env)


In [ ]:
with progress.stage("05-safety-validation-after-sft"):
    safety_data=project/"data/processed/safety-pilot-v1/validation.jsonl"
    safety_predictions=project/"runs/safety-sft-100-v1/predictions.jsonl"
    progress.run([gpu_py,"-u","scripts/generate.py","--model","CohereLabs/tiny-aya-global",
        "--revision",global_sha,"--adapter",str(sft_dir),"--data",str(safety_data),"--condition","finance-sft-100-safety-v1",
        "--out",str(safety_predictions),"--max-new-tokens","256","--load-in-4bit","--resume"],cwd=project,env=runtime_env)


In [ ]:
with progress.stage("06-review-packet"):
    from finsafe.progress import atomic_json
    review_packet=project/"runs/safety-sft-100-v1/review.jsonl"
    progress.run([gpu_py,"scripts/review_packet.py","--data",str(safety_data),"--predictions",str(safety_predictions),"--out",str(review_packet)],cwd=project,env=runtime_env)
    predictions=read_jsonl(safety_predictions)
    capped=sum(p["output_tokens"]>=256 for p in predictions)
    atomic_json(project/"runs/safety-sft-100-v1/status.json",{"n":len(predictions),"cap_hits":capped,"cap":256,"safety_rates":None,"label_status":"UNREVIEWED"})
    progress.log(f"Safety baseline saved: {len(predictions)} responses; {capped} hit the output cap. Inspect capped responses before labeling.")


In [ ]:
with progress.stage("07-review-and-recovery-backup"):
    import zipfile
    from IPython.display import FileLink,display
    backup=Path("/kaggle/working/finsafe-stage05-recovery.zip")
    files=[f for folder in [project/"data/processed",project/"runs",sft_dir] for f in folder.rglob("*") if f.is_file() and not f.is_symlink() and not f.name.startswith(".env")]
    progress.log(f"Backing up {len(files)} files ({sum(f.stat().st_size for f in files)/2**20:.1f} MiB)")
    with zipfile.ZipFile(backup,"w",compression=zipfile.ZIP_STORED) as archive:
        for i,file in enumerate(files,1):
            archive.write(file,str(file.relative_to(project)))
            if i%25==0 or i==len(files):progress.log(f"BACKUP {i}/{len(files)}")
    with zipfile.ZipFile(backup) as archive:
        if archive.testzip():raise RuntimeError("Recovery ZIP failed integrity check")
    display(FileLink(str(backup)))
    display(FileLink(str(review_packet)))
    progress.log("Stage 05 complete. Safety responses require review; no safety rates are claimed.")
